# Gopi TEC SJSP analysis - December/2024

In [ ]:
#Download Gopi_TEC_SJSP_Dec_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown ''

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'Gopi_TEC_SJSP_Dec_2024.zip'

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import glob

data_dir = '.'

std_files = glob.glob(os.path.join(data_dir, 'sjsp3*-2024-12-*.Std'))

std_files.sort()

dfs = []

for file_path in std_files:
    try:

        filename = os.path.basename(file_path)
        print(f"Processing {filename}")

        date_parts = filename.replace('.Std', '').split('-')
        year = int(date_parts[1])
        month = int(date_parts[2])
        day = int(date_parts[3])
        base_date = datetime(year, month, day)

        df = pd.read_csv(file_path, sep='\s+', header=None)

        df.columns = ['time_ut', 'tec', 'tec_std', 'latitude']

        def decimal_to_time(decimal_hours):
            hours = int(decimal_hours)
            minutes = int((decimal_hours - hours) * 60)
            seconds = int(((decimal_hours - hours) * 60 - minutes) * 60)
            return base_date + timedelta(hours=hours, minutes=minutes, seconds=seconds)

        df['DATETIME'] = df['time_ut'].apply(decimal_to_time)

        df['tec'] = pd.to_numeric(df['tec'].replace('-', np.nan))
        df['tec_std'] = pd.to_numeric(df['tec_std'].replace('-', np.nan))

        df = df[['DATETIME', 'tec', 'tec_std', 'latitude']]
        df.columns = ['DATETIME', 'TEC', 'TEC_STD', 'LATITUDE']

        dfs.append(df)

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

    combined_df = combined_df.sort_values('DATETIME')

    combined_df.to_pickle('sjsp_dec_2024_complete.pkl')

    print(f"Combined DataFrame created with {len(combined_df)} rows")
    print("First 5 lines:")
    print(combined_df.head())

    print("\nPeriod covered by data:")
    print(f"Start: {combined_df['DATETIME'].min()}")
    print(f"End: {combined_df['DATETIME'].max()}")
    print(f"Total days: {(combined_df['DATETIME'].max() - combined_df['DATETIME'].min()).days + 1}")

    present_days = combined_df['DATETIME'].dt.date.unique()
    print(f"\nTotal days with data: {len(present_days)}")
    print("Dias presentes:", sorted(present_days))
else:
    print(f"\nTotal days with data: {len(present_days)}")

In [ ]:
combined_df

## MIN/MAX

In [ ]:
# import pandas as pd
# import numpy as np
# from datetime import timedelta, time, datetime

# df = pd.read_pickle('sjsp_dec_2024_complete.pkl')
# print(f"Data loaded: {len(df)} records of {df['DATETIME'].min().date()} until {df['DATETIME'].max().date()}")

# print(f"Available columns in the DataFrame: {list(df.columns)}")

# df = df.sort_values('DATETIME').reset_index(drop=True)

# df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
# print(f"Analysing {len(df_clean)} points after removing missing values")

# def check_consecutive_highs(df, peak_datetime, threshold_percent=90):
#     peak_idx = df[df['DATETIME'] == peak_datetime].index[0]
#     peak_value = df.loc[peak_idx, 'TEC']
#     threshold = peak_value * (threshold_percent / 100)

#     idx_before = peak_idx - 1
#     idx_after = peak_idx + 1

#     valid_before = idx_before >= 0
#     valid_after = idx_after < len(df)

#     if valid_before and valid_after:
#         values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
#         stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
#         datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [idx_before, peak_idx, idx_after],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     if valid_before and idx_before > 0:
#         values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
#         stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
#         datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [idx_before-1, idx_before, peak_idx],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     if valid_after and idx_after < len(df) - 1:
#         values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
#         stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
#         datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [peak_idx, idx_after, idx_after+1],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     return None

# def find_closest_value_to_datetime(df, target_datetime, max_minutes_diff=2):

#     if isinstance(target_datetime, str):
#         target_datetime = pd.to_datetime(target_datetime)
#     elif isinstance(target_datetime, datetime):
#         target_datetime = pd.Timestamp(target_datetime)

#     time_diff = abs((df['DATETIME'] - target_datetime).dt.total_seconds() / 60)

#     within_range = time_diff <= max_minutes_diff

#     if not within_range.any():
#         return None, None

#     closest_idx = time_diff[within_range].idxmin()
#     closest_diff = time_diff[closest_idx]

#     return df.loc[closest_idx], closest_diff

# def get_available_location_info(row):

#     location_info = {}

#     location_columns = ['LATITUDE', 'LONGITUDE', 'LAT', 'LON', 'STATION', 'SITE']

#     for col in location_columns:
#         if col in row.index:
#             location_info[col] = row[col]

#     return location_info

# def analyze_specific_datetimes(target_datetimes, max_minutes_diff=2):

#     print(f"\n{'='*100}")
#     print(f"ANALYSIS OF TEC VALUES CLOSE TO SPECIFIC TIMES")
#     print(f"(Maximum allowed difference: {max_minutes_diff} minutes)")
#     print(f"{'='*100}")

#     results = []

#     for i, target_dt in enumerate(target_datetimes):
#         print(f"\n{'-'*60}")
#         print(f"LOOKING FOR VALUES NEAR: {target_dt}")
#         print(f"{'-'*60}")

#         closest_row, time_diff = find_closest_value_to_datetime(df_clean, target_dt, max_minutes_diff)

#         if closest_row is not None:
#             print(f" VALUE FOUND:")
#             print(f" Actual Date/Time: {closest_row['DATETIME']}")
#             print(f" Target Date/Time: {target_dt}")
#             print(f" Difference: {time_diff:.2f} minutes")
#             print(f" TEC: {closest_row['TEC']:.2f} ± {closest_row['TEC_STD']:.2f} TECU")

#             location_info = get_available_location_info(closest_row)
#             for key, value in location_info.items():
#                 print(f"  {key}: {value}")

#             sequence = check_consecutive_highs(df_clean, closest_row['DATETIME'], threshold_percent=90)

#             if sequence:
#                 print(f" HAS a sequence of 3 high values ​​(≥90% of the peak):")
#                 for j, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
#                     print(f"    {j+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/closest_row['TEC']*100:.1f}% of the value)")
#             else:
#                 print(f" does NOT have a sequence of 3 consecutive high values")

#             result = {
#                 'target_datetime': target_dt,
#                 'found_datetime': closest_row['DATETIME'],
#                 'time_diff_minutes': time_diff,
#                 'TEC': closest_row['TEC'],
#                 'TEC_STD': closest_row['TEC_STD'],
#                 'has_sequence': sequence is not None,
#                 'sequence': sequence,
#                 'location_info': location_info
#             }

#             results.append(result)

#         else:
#             print(f" NO VALUE FOUND within {max_minutes_diff} minutes of target time")
#             results.append({
#                 'target_datetime': target_dt,
#                 'found_datetime': None,
#                 'time_diff_minutes': None,
#                 'TEC': None,
#                 'TEC_STD': None,
#                 'has_sequence': False,
#                 'sequence': None,
#                 'location_info': {}
#             })

#     print(f"\n{'='*120}")
#     print(f"RESULTS SUMMARY TABLE")
#     print(f"{'='*120}")

#     header = f"{'#':<3}{'Target Time':<20}{'Found Time':<20}{'Diff(min)':<10}{'TEC':<10}{'TEC_STD':<10}{'Sequence':<12}"

#     location_cols = set()
#     for result in results:
#         if result['location_info']:
#             location_cols.update(result['location_info'].keys())

#     for col in sorted(location_cols):
#         header += f"{col:<12}"

#     print(header)
#     print(f"{'-'*len(header)}")

#     for i, result in enumerate(results):
#         if result['found_datetime'] is not None:
#             seq_info = "Sim" if result['has_sequence'] else "Não"
#             row = f"{i+1:<3}{str(result['target_datetime']):<20}{str(result['found_datetime']):<20}"
#             row += f"{result['time_diff_minutes']:.2f}    {result['TEC']:.2f}    "
#             row += f"{result['TEC_STD']:.2f}    {seq_info:<12}"

#             for col in sorted(location_cols):
#                 value = result['location_info'].get(col, 'N/A')
#                 if isinstance(value, float):
#                     row += f"{value:.2f}      "
#                 else:
#                     row += f"{str(value):<12}"

#             print(row)
#         else:
#             row = f"{i+1:<3}{str(result['target_datetime']):<20}{'NÃO ENCONTRADO':<20}"
#             row += f"{'N/A':<10}{'N/A':<10}{'N/A':<10}{'N/A':<12}"

#             for col in sorted(location_cols):
#                 row += f"{'N/A':<12}"

#             print(row)

#     print(f"{'-'*len(header)}")

#     return results

# target_datetimes = [
#     "2024-12-18 18:20:00",
#     "2024-12-18 18:10:00",
#     "2024-12-18 18:30:00",
#     "2024-12-18 18:00:00"
# ]

# print("\n\n" + "*"*50 + " ANALYSIS OF SPECIFIC TIMES " + "*"*50)
# results = analyze_specific_datetimes(target_datetimes, max_minutes_diff=2)

# print("\nAnalysis complete!")

# found_results = [r for r in results if r['found_datetime'] is not None]
# if found_results:
#     print(f"\n{'='*80}")
#     print(f"STATISTICS OF THE RESULTS FOUND")
#     print(f"{'='*80}")

#     tec_values = [r['TEC'] for r in found_results]
#     print(f"TEC values ​​found:")
#     print(f" Minimum: {min(tec_values):.2f} TECU")
#     print(f" Maximum: {max(tec_values):.2f} TECU")
#     print(f" Mean: {np.mean(tec_values):.2f} TECU")
#     print(f" Standard deviation: {np.std(tec_values):.2f} TECU")

#     sequences_found = sum(1 for r in found_results if r['has_sequence'])
#     print(f"\nHigh value sequences found: {sequences_found} out of {len(found_results)} ({sequences_found/len(found_results)*100:.1f}%)")

## 18:00

In [ ]:
# import pandas as pd
# import numpy as np
# from datetime import timedelta, time, datetime

# df = pd.read_pickle('sjsp_dec_2024_complete.pkl')
# print(f"Data loaded: {len(df)} records of {df['DATETIME'].min().date()} until {df['DATETIME'].max().date()}")

# print(f"Available columns in the DataFrame: {list(df.columns)}")

# df = df.sort_values('DATETIME').reset_index(drop=True)

# df_clean = df.dropna(subset=['TEC', 'TEC_STD']).copy()
# print(f"Analysing {len(df_clean)} points after removing missing values")

# def check_consecutive_highs(df, peak_datetime, threshold_percent=90):
#     peak_idx = df[df['DATETIME'] == peak_datetime].index[0]
#     peak_value = df.loc[peak_idx, 'TEC']
#     threshold = peak_value * (threshold_percent / 100)

#     idx_before = peak_idx - 1
#     idx_after = peak_idx + 1

#     valid_before = idx_before >= 0
#     valid_after = idx_after < len(df)

#     if valid_before and valid_after:
#         values = [df.loc[idx_before, 'TEC'], peak_value, df.loc[idx_after, 'TEC']]
#         stds = [df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD']]
#         datetimes = [df.loc[idx_before, 'DATETIME'], peak_datetime, df.loc[idx_after, 'DATETIME']]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [idx_before, peak_idx, idx_after],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     if valid_before and idx_before > 0:
#         values = [df.loc[idx_before-1, 'TEC'], df.loc[idx_before, 'TEC'], peak_value]
#         stds = [df.loc[idx_before-1, 'TEC_STD'], df.loc[idx_before, 'TEC_STD'], df.loc[peak_idx, 'TEC_STD']]
#         datetimes = [df.loc[idx_before-1, 'DATETIME'], df.loc[idx_before, 'DATETIME'], peak_datetime]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [idx_before-1, idx_before, peak_idx],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     if valid_after and idx_after < len(df) - 1:
#         values = [peak_value, df.loc[idx_after, 'TEC'], df.loc[idx_after+1, 'TEC']]
#         stds = [df.loc[peak_idx, 'TEC_STD'], df.loc[idx_after, 'TEC_STD'], df.loc[idx_after+1, 'TEC_STD']]
#         datetimes = [peak_datetime, df.loc[idx_after, 'DATETIME'], df.loc[idx_after+1, 'DATETIME']]
#         all_high = all(value >= threshold for value in values)

#         if all_high:
#             return {'indices': [peak_idx, idx_after, idx_after+1],
#                    'values': values,
#                    'stds': stds,
#                    'datetimes': datetimes}

#     return None

# def find_closest_value_to_datetime(df, target_datetime, max_minutes_diff=2):

#     if isinstance(target_datetime, str):
#         target_datetime = pd.to_datetime(target_datetime)
#     elif isinstance(target_datetime, datetime):
#         target_datetime = pd.Timestamp(target_datetime)

#     time_diff = abs((df['DATETIME'] - target_datetime).dt.total_seconds() / 60)

#     within_range = time_diff <= max_minutes_diff

#     if not within_range.any():
#         return None, None

#     closest_idx = time_diff[within_range].idxmin()
#     closest_diff = time_diff[closest_idx]

#     return df.loc[closest_idx], closest_diff

# def get_available_location_info(row):

#     location_info = {}

#     location_columns = ['LATITUDE', 'LONGITUDE', 'LAT', 'LON', 'STATION', 'SITE']

#     for col in location_columns:
#         if col in row.index:
#             location_info[col] = row[col]

#     return location_info

# def analyze_specific_datetimes(target_datetimes, max_minutes_diff=2):

#     print(f"\n{'='*100}")
#     print(f"ANALYSIS OF TEC VALUES CLOSE TO SPECIFIC TIMES")
#     print(f"(Maximum allowed difference: {max_minutes_diff} minutes)")
#     print(f"{'='*100}")

#     results = []

#     for i, target_dt in enumerate(target_datetimes):
#         print(f"\n{'-'*60}")
#         print(f"LOOKING FOR VALUES NEAR: {target_dt}")
#         print(f"{'-'*60}")

#         closest_row, time_diff = find_closest_value_to_datetime(df_clean, target_dt, max_minutes_diff)

#         if closest_row is not None:
#             print(f" VALUE FOUND:")
#             print(f" Actual Date/Time: {closest_row['DATETIME']}")
#             print(f" Target Date/Time: {target_dt}")
#             print(f" Difference: {time_diff:.2f} minutes")
#             print(f"  TEC: {closest_row['TEC']:.2f} ± {closest_row['TEC_STD']:.2f} TECU")

#             location_info = get_available_location_info(closest_row)
#             for key, value in location_info.items():
#                 print(f"  {key}: {value}")

#             sequence = check_consecutive_highs(df_clean, closest_row['DATETIME'], threshold_percent=90)

#             if sequence:
#                 print(f" HAS a sequence of 3 high values ​​(≥90% of the peak):")
#                 for j, (dt, val, std) in enumerate(zip(sequence['datetimes'], sequence['values'], sequence['stds'])):
#                     print(f"    {j+1}. {dt}: {val:.2f} ± {std:.2f} TECU ({val/closest_row['TEC']*100:.1f}% of the value)")
#             else:
#                 print(f" does NOT have a sequence of 3 consecutive high values")

#             result = {
#                 'target_datetime': target_dt,
#                 'found_datetime': closest_row['DATETIME'],
#                 'time_diff_minutes': time_diff,
#                 'TEC': closest_row['TEC'],
#                 'TEC_STD': closest_row['TEC_STD'],
#                 'has_sequence': sequence is not None,
#                 'sequence': sequence,
#                 'location_info': location_info
#             }

#             results.append(result)

#         else:
#             print(f"NO VALUE FOUND within {max_minutes_diff} minutes of target time")
#             results.append({
#                 'target_datetime': target_dt,
#                 'found_datetime': None,
#                 'time_diff_minutes': None,
#                 'TEC': None,
#                 'TEC_STD': None,
#                 'has_sequence': False,
#                 'sequence': None,
#                 'location_info': {}
#             })

#     print(f"\n{'='*120}")
#     print(f"RESULTS SUMMARY TABLE")
#     print(f"{'='*120}")

#     header = f"{'#':<3}{'Target Time':<20}{'Found Time':<20}{'Diff(min)':<10}{'TEC':<10}{'TEC_STD':<10}{'Sequence':<12}"

#     location_cols = set()
#     for result in results:
#         if result['location_info']:
#             location_cols.update(result['location_info'].keys())

#     for col in sorted(location_cols):
#         header += f"{col:<12}"

#     print(header)
#     print(f"{'-'*len(header)}")

#     for i, result in enumerate(results):
#         if result['found_datetime'] is not None:
#             seq_info = "Yes" if result['has_sequence'] else "No"
#             row = f"{i+1:<3}{str(result['target_datetime']):<20}{str(result['found_datetime']):<20}"
#             row += f"{result['time_diff_minutes']:.2f}    {result['TEC']:.2f}    "
#             row += f"{result['TEC_STD']:.2f}    {seq_info:<12}"

#             for col in sorted(location_cols):
#                 value = result['location_info'].get(col, 'N/A')
#                 if isinstance(value, float):
#                     row += f"{value:.2f}      "
#                 else:
#                     row += f"{str(value):<12}"

#             print(row)
#         else:
#             row = f"{i+1:<3}{str(result['target_datetime']):<20}{'NOT FOUND':<20}"
#             row += f"{'N/A':<10}{'N/A':<10}{'N/A':<10}{'N/A':<12}"

#             for col in sorted(location_cols):
#                 row += f"{'N/A':<12}"

#             print(row)

#     print(f"{'-'*len(header)}")

#     return results

# target_datetimes = [
#     "2024-12-01 18:00:00"
# ]

# print("\n\n" + "*"*50 + " ANALYSIS OF SPECIFIC TIMES " + "*"*50)
# results = analyze_specific_datetimes(target_datetimes, max_minutes_diff=2)

# print("\nAnalysis complete!")

# found_results = [r for r in results if r['found_datetime'] is not None]
# if found_results:
#     print(f"\n{'='*80}")
#     print(f"STATISTICS OF THE RESULTS FOUND")
#     print(f"{'='*80}")

#     tec_values = [r['TEC'] for r in found_results]
#     print(f"TEC values ​​found:")
#     print(f" Minimum: {min(tec_values):.2f} TECU")
#     print(f" Maximum: {max(tec_values):.2f} TECU")
#     print(f" Mean: {np.mean(tec_values):.2f} TECU")
#     print(f" Standard deviation: {np.std(tec_values):.2f} TECU")

#     sequences_found = sum(1 for r in found_results if r['has_sequence'])
#     print(f"\nHigh value sequences found: {sequences_found} out of {len(found_results)} ({sequences_found/len(found_results)*100:.1f}%)")

# TEC maps analysis

In [ ]:
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points

## EMBRACE

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1LkY0IeFfdnahoH81OKYW-2xmXHgjFPhH'

In [ ]:
df_embrace_maps_2024 = pd.read_pickle('/content/TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_embrace_maps_2024

In [ ]:
class Embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
embrace_tec = Embrace()
embrace_points = embrace_tec.find_nearest_points(lat=-23.20713, lon=-45.86174)

In [ ]:
embrace_points

### MIN

In [ ]:
# import pandas as pd

# date = '2024-12-18'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_igs_maps_2024[df_igs_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:

#     df_temp = df_igs_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_igs_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_igs_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
igs_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_igs_maps = []
for i in range(len(igs_maps)):
    np_igs_maps.append(igs_maps[i])
np_igs_maps = np.array(np_igs_maps)

In [ ]:
np_igs_maps

In [ ]:
np_igs_maps.shape

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

### MAX

In [ ]:
# import pandas as pd

# date = '2024-12-19'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_igs_maps_2024[df_igs_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_igs_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_igs_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_igs_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
igs_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_igs_maps = []
for i in range(len(igs_maps)):
    np_igs_maps.append(igs_maps[i])
np_igs_maps = np.array(np_igs_maps)

In [ ]:
np_igs_maps

In [ ]:
np_igs_maps.shape

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## IGS

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1T0gBaH6IyIX72T1VPJuZa2D43HCrRFVp'

In [ ]:
df_igs_maps_2024 = pd.read_pickle('/content/TF_IGS_TEC_maps_intersection_case_study_Dec_2024.pkl')

In [ ]:
df_igs_maps_2024

In [ ]:
class Igs(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),  # (lon_min, lon_max, lat_min_geo, lat_max_geo)
                 lat_step: float = 2.5,
                 lon_step: float = 5):
        """
        IGS TEC Map - Specific implementation for IGS data structure.

        IMPORTANT: IGS arrays are organized North to South:
        - Array[0, :] contains the NORTHERNMOST latitudes (+40°)
        - Array[-1, :] contains the SOUTHERNMOST latitudes (-80°)

        The extent parameter follows GEOGRAPHIC convention:
        - extent = (lon_min, lon_max, lat_min_geographic, lat_max_geographic)
        - lat_min_geographic = -80° (southernmost point)
        - lat_max_geographic = +40° (northernmost point)

        This differs from array indexing where Array[0] = +40° (North).
        The find_nearest_points method handles this coordinate conversion.
        """
        super().__init__(extent, lat_step, lon_step)

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat_max - lat) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_max - lat_idx_lower * lat_step
        lat_upper = lat_max - lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points

In [ ]:
igs_tec = Igs()
igs_points = igs_tec.find_nearest_points(lat=-23.20713, lon=-45.86174)

In [ ]:
igs_points

### MIN

In [ ]:
# import pandas as pd

# date = '2024-12-18'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_igs_maps_2024[df_igs_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:

#     df_temp = df_igs_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_igs_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_igs_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
igs_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_igs_maps = []
for i in range(len(igs_maps)):
    np_igs_maps.append(igs_maps[i])
np_igs_maps = np.array(np_igs_maps)

In [ ]:
np_igs_maps

In [ ]:
np_igs_maps.shape

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

### MAX

In [ ]:
# import pandas as pd

# date = '2024-12-19'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_igs_maps_2024[df_igs_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_igs_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_igs_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_igs_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
igs_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_igs_maps = []
for i in range(len(igs_maps)):
    np_igs_maps.append(igs_maps[i])
np_igs_maps = np.array(np_igs_maps)

In [ ]:
np_igs_maps

In [ ]:
np_igs_maps.shape

In [ ]:
igs_tec.tec_map = np_igs_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(igs_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = igs_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in igs_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(igs_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in igs_points]
lons = [p[1] for p in igs_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(igs_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## MAGGIA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1Qi7Xm93lhuwGWQnr3N-gquhzzUxegI33'

In [ ]:
!curl 'https://drive.usercontent.google.com/download?id=1Qi7Xm93lhuwGWQnr3N-gquhzzUxegI33&export=download&authuser=0&confirm=t&uuid=c08ac5de-3885-4f79-94ca-8c330c18a009&at=AN8xHoquUkJpNDx3MgFLBpAklXfq%3A1758216852519' \
  -H 'accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7' \
  -H 'accept-language: pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7' \
  -H 'cache-control: no-cache' \
  -b 'SID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE2PPdTeaOJdBeAtlpmXw0sewACgYKAbcSARMSFQHGX2MiN1EGMoYW5-zZxBT1_ezSjxoVAUF8yKp-Bxf-8e594r0CQzdCemGa0076; __Secure-1PSID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE232KZqePrzsSMgAiOjp4augACgYKAfwSARMSFQHGX2MiPmCqH2pVXTz0Lo82CmWsBhoVAUF8yKr1aTm8Qb3W38L98ZBLWPW00076; __Secure-3PSID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE2breUz8eJRfYGoHI-qm-VKQACgYKAagSARMSFQHGX2Mi-twVlgPzsTmjh9L3oX7OnRoVAUF8yKoG1gSkhWePQUxvPuKDbr8S0076; HSID=AMSoVtm4f2JOblFWQ; SSID=Ar4xkmiXaV7ZvsP_s; APISID=kylgo9NxpVDz00IS/AjUM8Yk5s56Ua4Eh8; SAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; __Secure-1PAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; __Secure-3PAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; AEC=AaJma5sHSMxOHlxGgF7UlW_iRFyPyjgEr18diEgqqKCL2BLcwan5fEPRB2s; NID=525=CNtlUvsSnUt4R6c7ftiXrWGpj-HywIsZMApOnrSLpatxi1U5dgZBC5UHUQzq7Pjps-73Xed-NUfBQ7OYju3vKKs7V4Qw_7yB7oHkcD1p2V3c3pX7-MiOZUiJHmSkgfa1pQCwfANSpSjelh8Ndyh6mEXWPm3-gbvVVJlQaowIOKd_57vTOsMYVT14LVIJKdqekPmvS7pY19rFTlH0KMR1qAKmMToSum6NmVfk_p6FgAMGTbTQPDkZadOC-GJsoITTxuMz6pq2UxqArcy3IUzb5uQCbnHwUDaPUlJueqCjwUsnmQjx5_LdWMGRMUdwmNr4fckZRsQtrmM4y4NKg-eMQ7I4GQ2Pf9k6oW0uec99qvGGf6djhOB5EymsROT6II3w6prjjqxe9zODQVxTHx8rsWUjCm9ch_x6EeuYb9I0eNVs7NszzhNHDbT_PnuiNhx2fdxkRDLx3aG3pv4QxJKpbQu8FhEAbp7BmHEn1Y9EWsM1NkPf3eMVd_XEY6ZqZuJRdLORbg-6aDI0TXn3FG2K0zbJuSCKaAN3w0mQ9eb-fSS_UOe34iFWC8jBJVSExm_bZm1DM0u5VKdRqyyzeLvaapxcoq1Yr42pmn7JRVKhV47kCQjjs6zUKBiAp8U1wUrJF0ARe060TY-sZG7qbmbqmeLP87ykzXsb2Agr1GwA6vI1E-Y_EnSsKS7soFLQAx5C2yCOwRcp9EnjUl11cweqkc99_TGeESvGKjFbxcBVNRia3zHJdZcbDffn--cII9HZmOClyv2nrlU-MuIYugpTIhoyPCU71TihKMuJvv_sduOJ0aTzYWwDkQsdnmNk-FG7ynOzC2OCJaHfuGtAt8Z8h2Fr_8Do32Gy5tPqeACm39QPKhD9Z1W-azqpwChPKAG_rWZfNa6kjYCTBKCx0VnCgx2uTgeCP2GFZM9K; OSID=g.a0001ggDnh5vx7v2eXlOltr9BhXJTuJGu2rYsDhPqgRqu_bMlLhWSwgL10CN_GVXM8AAGTOfjQACgYKAc8SARMSFQHGX2Mio9qh9iwu6yV724Fq_YOEaBoVAUF8yKrkBz3OaQOngPuEPxIRQLYk0076; __Secure-OSID=g.a0001ggDnh5vx7v2eXlOltr9BhXJTuJGu2rYsDhPqgRqu_bMlLhWDP1aRXplTvdXAt02bUhtHwACgYKAccSARMSFQHGX2MiwPavYoxhd3f5_lXW-LBmNhoVAUF8yKrk8442K3HPIhUbccBs4W2a0076; __Secure-1PSIDTS=sidts-CjIBmkD5SyLITgP9_sYOQuEI9JJ2RgtchvpW1-RF2yiJLgz2b9da-ciV_5aN2Vkn1r94mBAA; __Secure-3PSIDTS=sidts-CjIBmkD5SyLITgP9_sYOQuEI9JJ2RgtchvpW1-RF2yiJLgz2b9da-ciV_5aN2Vkn1r94mBAA; SIDCC=AKEyXzXuaje9M0q4LWLp1nGjS2LA2sAonEPAj0AMsIHNMw8y0rmkJS-XdojNu-FOa9AwUNZMUtM; __Secure-1PSIDCC=AKEyXzUrezOPnzm1BJBcxjq3B1YEnB0N-6Xi82Yc95wzIiHbKnEqcTn7Em0kvhMZfFVMP8Te1w; __Secure-3PSIDCC=AKEyXzXq6KHNyRMkrqVDBLZhU5zQRcg6zMUup2Uzc_G3vLmPBx8v8jpTHdJuySnfX767xF5kBg' \
  -H 'pragma: no-cache' \
  -H 'priority: u=0, i' \
  -H 'sec-ch-ua: "Chromium";v="140", "Not=A?Brand";v="24", "Google Chrome";v="140"' \
  -H 'sec-ch-ua-arch: "x86"' \
  -H 'sec-ch-ua-bitness: "64"' \
  -H 'sec-ch-ua-form-factors: "Desktop"' \
  -H 'sec-ch-ua-full-version: "140.0.7339.127"' \
  -H 'sec-ch-ua-full-version-list: "Chromium";v="140.0.7339.127", "Not=A?Brand";v="24.0.0.0", "Google Chrome";v="140.0.7339.127"' \
  -H 'sec-ch-ua-mobile: ?0' \
  -H 'sec-ch-ua-model: ""' \
  -H 'sec-ch-ua-platform: "Linux"' \
  -H 'sec-ch-ua-platform-version: "6.11.0"' \
  -H 'sec-ch-ua-wow64: ?0' \
  -H 'sec-fetch-dest: document' \
  -H 'sec-fetch-mode: navigate' \
  -H 'sec-fetch-site: cross-site' \
  -H 'sec-fetch-user: ?1' \
  -H 'upgrade-insecure-requests: 1' \
  -H 'user-agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36' \
  -H 'x-browser-channel: stable' \
  -H 'x-browser-copyright: Copyright 2025 Google LLC. All rights reserved.' \
  -H 'x-browser-validation: vkog5hSrCvuYLnOVCb84JEvPqcg=' \
  -H 'x-browser-year: 2025' \
  -H 'x-client-data: CJO2yQEIo7bJAQipncoBCNCgygEIrYfLAQiTocsBCIegzQE=' > 'TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl'

In [ ]:
df_maggia_maps_2024 = pd.read_pickle('/content/TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_maggia_maps_2024

In [ ]:
class Maggia(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
maggia_tec = Maggia()
maggia_points = maggia_tec.find_nearest_points(lat=-23.20713, lon=-45.86174)

In [ ]:
maggia_points

### MIN

In [ ]:
# import pandas as pd

# date = '2024-12-18'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_maggia_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_maggia_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

### MAX

In [ ]:
# import pandas as pd

# date = '2024-12-18'
# time = '18:10:00'
# timestamp_wanted = f'{date} {time}'

# result = df_maggia_maps_2024[df_maggia_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_maggia_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_maggia_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_maggia_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
maggia_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_maggia_maps = []
for i in range(len(maggia_maps)):
    np_maggia_maps.append(maggia_maps[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

## Nagoya

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_Nagoya_TEC_maps_intersection_case_study_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
!gdown '1CXL-btSyeELqYjLGL-B2oSnMt-QlPFUR'

In [ ]:
!curl 'https://drive.usercontent.google.com/download?id=1CXL-btSyeELqYjLGL-B2oSnMt-QlPFUR&export=download&authuser=0&confirm=t&uuid=e62294f3-5d3b-4201-9821-f868319ab167&at=AN8xHorzxZ0OGZmSGDgcRLrJ2nEF%3A1758217090390' \
  -H 'accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7' \
  -H 'accept-language: pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7' \
  -H 'cache-control: no-cache' \
  -b 'SID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE2PPdTeaOJdBeAtlpmXw0sewACgYKAbcSARMSFQHGX2MiN1EGMoYW5-zZxBT1_ezSjxoVAUF8yKp-Bxf-8e594r0CQzdCemGa0076; __Secure-1PSID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE232KZqePrzsSMgAiOjp4augACgYKAfwSARMSFQHGX2MiPmCqH2pVXTz0Lo82CmWsBhoVAUF8yKr1aTm8Qb3W38L98ZBLWPW00076; __Secure-3PSID=g.a0001QgDnv7PqHbM8Pn7sIxqKj3lHVD4fl3G2RgkdSiAccVawJE2breUz8eJRfYGoHI-qm-VKQACgYKAagSARMSFQHGX2Mi-twVlgPzsTmjh9L3oX7OnRoVAUF8yKoG1gSkhWePQUxvPuKDbr8S0076; HSID=AMSoVtm4f2JOblFWQ; SSID=Ar4xkmiXaV7ZvsP_s; APISID=kylgo9NxpVDz00IS/AjUM8Yk5s56Ua4Eh8; SAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; __Secure-1PAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; __Secure-3PAPISID=dfw72HSW2gMI0NIv/ActmezOe8M9c6ZNKq; AEC=AaJma5sHSMxOHlxGgF7UlW_iRFyPyjgEr18diEgqqKCL2BLcwan5fEPRB2s; NID=525=CNtlUvsSnUt4R6c7ftiXrWGpj-HywIsZMApOnrSLpatxi1U5dgZBC5UHUQzq7Pjps-73Xed-NUfBQ7OYju3vKKs7V4Qw_7yB7oHkcD1p2V3c3pX7-MiOZUiJHmSkgfa1pQCwfANSpSjelh8Ndyh6mEXWPm3-gbvVVJlQaowIOKd_57vTOsMYVT14LVIJKdqekPmvS7pY19rFTlH0KMR1qAKmMToSum6NmVfk_p6FgAMGTbTQPDkZadOC-GJsoITTxuMz6pq2UxqArcy3IUzb5uQCbnHwUDaPUlJueqCjwUsnmQjx5_LdWMGRMUdwmNr4fckZRsQtrmM4y4NKg-eMQ7I4GQ2Pf9k6oW0uec99qvGGf6djhOB5EymsROT6II3w6prjjqxe9zODQVxTHx8rsWUjCm9ch_x6EeuYb9I0eNVs7NszzhNHDbT_PnuiNhx2fdxkRDLx3aG3pv4QxJKpbQu8FhEAbp7BmHEn1Y9EWsM1NkPf3eMVd_XEY6ZqZuJRdLORbg-6aDI0TXn3FG2K0zbJuSCKaAN3w0mQ9eb-fSS_UOe34iFWC8jBJVSExm_bZm1DM0u5VKdRqyyzeLvaapxcoq1Yr42pmn7JRVKhV47kCQjjs6zUKBiAp8U1wUrJF0ARe060TY-sZG7qbmbqmeLP87ykzXsb2Agr1GwA6vI1E-Y_EnSsKS7soFLQAx5C2yCOwRcp9EnjUl11cweqkc99_TGeESvGKjFbxcBVNRia3zHJdZcbDffn--cII9HZmOClyv2nrlU-MuIYugpTIhoyPCU71TihKMuJvv_sduOJ0aTzYWwDkQsdnmNk-FG7ynOzC2OCJaHfuGtAt8Z8h2Fr_8Do32Gy5tPqeACm39QPKhD9Z1W-azqpwChPKAG_rWZfNa6kjYCTBKCx0VnCgx2uTgeCP2GFZM9K; OSID=g.a0001ggDnh5vx7v2eXlOltr9BhXJTuJGu2rYsDhPqgRqu_bMlLhWSwgL10CN_GVXM8AAGTOfjQACgYKAc8SARMSFQHGX2Mio9qh9iwu6yV724Fq_YOEaBoVAUF8yKrkBz3OaQOngPuEPxIRQLYk0076; __Secure-OSID=g.a0001ggDnh5vx7v2eXlOltr9BhXJTuJGu2rYsDhPqgRqu_bMlLhWDP1aRXplTvdXAt02bUhtHwACgYKAccSARMSFQHGX2MiwPavYoxhd3f5_lXW-LBmNhoVAUF8yKrk8442K3HPIhUbccBs4W2a0076; __Secure-1PSIDTS=sidts-CjIBmkD5SyLITgP9_sYOQuEI9JJ2RgtchvpW1-RF2yiJLgz2b9da-ciV_5aN2Vkn1r94mBAA; __Secure-3PSIDTS=sidts-CjIBmkD5SyLITgP9_sYOQuEI9JJ2RgtchvpW1-RF2yiJLgz2b9da-ciV_5aN2Vkn1r94mBAA; SIDCC=AKEyXzWFvJYTVOWwklbps7CGYj_lvH__oFCd0zQYe0WtTJmitgFN09568k_O5-gjuWvVeadmeT4; __Secure-1PSIDCC=AKEyXzXwJvBZ6vV2H9Ua9mRN_9KX71xQsaSI38-F62LFPYHzCaqbYn3pAgr776EswU0vpsBSxg; __Secure-3PSIDCC=AKEyXzW1IqN__lDZ8oR8EbYqjahtmuYEtw_SD-LwyR6-suKZ7eIYLm8N-tMCroVUrWPhquluZQ' \
  -H 'pragma: no-cache' \
  -H 'priority: u=0, i' \
  -H 'sec-ch-ua: "Chromium";v="140", "Not=A?Brand";v="24", "Google Chrome";v="140"' \
  -H 'sec-ch-ua-arch: "x86"' \
  -H 'sec-ch-ua-bitness: "64"' \
  -H 'sec-ch-ua-form-factors: "Desktop"' \
  -H 'sec-ch-ua-full-version: "140.0.7339.127"' \
  -H 'sec-ch-ua-full-version-list: "Chromium";v="140.0.7339.127", "Not=A?Brand";v="24.0.0.0", "Google Chrome";v="140.0.7339.127"' \
  -H 'sec-ch-ua-mobile: ?0' \
  -H 'sec-ch-ua-model: ""' \
  -H 'sec-ch-ua-platform: "Linux"' \
  -H 'sec-ch-ua-platform-version: "6.11.0"' \
  -H 'sec-ch-ua-wow64: ?0' \
  -H 'sec-fetch-dest: document' \
  -H 'sec-fetch-mode: navigate' \
  -H 'sec-fetch-site: cross-site' \
  -H 'sec-fetch-user: ?1' \
  -H 'upgrade-insecure-requests: 1' \
  -H 'user-agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0.0.0 Safari/537.36' \
  -H 'x-browser-channel: stable' \
  -H 'x-browser-copyright: Copyright 2025 Google LLC. All rights reserved.' \
  -H 'x-browser-validation: vkog5hSrCvuYLnOVCb84JEvPqcg=' \
  -H 'x-browser-year: 2025' \
  -H 'x-client-data: CJO2yQEIo7bJAQipncoBCNCgygEIrYfLAQiTocsBCIegzQE=' > 'TF_Nagoya_TEC_maps_intersection_case_study_Dec_2024.pkl'

In [ ]:
df_nagoya_maps_2024 = pd.read_pickle('TF_Nagoya_TEC_maps_intersection_case_study_Dec_2024.pkl')

In [ ]:
df_nagoya_maps_2024

In [ ]:
class Nagoya(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80.4, 40.1),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
nagoya_tec = Nagoya()
nagoya_points = nagoya_tec.find_nearest_points(lat=-23.20713, lon=-45.86174)

In [ ]:
nagoya_points

### MIN

In [ ]:
# import pandas as pd

# date = '2024-12-01'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_nagoya_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_nagoya_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

# Value mapping
point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

### MAX

In [ ]:
# import pandas as pd

# date = '2024-12-18'
# time = '18:00:00'
# timestamp_wanted = f'{date} {time}'

# result = df_nagoya_maps_2024[df_nagoya_maps_2024['DATETIME'] == timestamp_wanted]

# if result.empty:
#     df_temp = df_nagoya_maps_2024.copy()
#     df_temp['DATETIME'] = pd.to_datetime(df_temp['DATETIME'])
#     timestamp_dt = pd.to_datetime(timestamp_wanted)

#     df_temp['diff'] = abs((df_temp['DATETIME'] - timestamp_dt).dt.total_seconds())
#     idx_nearest = df_temp['diff'].idxmin()

#     result = df_nagoya_maps_2024.iloc[[idx_nearest]]
#     print(f"Exact date not found. Using closest record: {df_nagoya_maps_2024.iloc[idx_nearest]['DATETIME']}")

In [ ]:
result

In [ ]:
nagoya_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_nagoya_maps = []
for i in range(len(nagoya_maps)):
    np_nagoya_maps.append(nagoya_maps[i])
np_nagoya_maps = np.array(np_nagoya_maps)

In [ ]:
np_nagoya_maps

In [ ]:
np_nagoya_maps.shape

In [ ]:
nagoya_tec.tec_map = np_nagoya_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(nagoya_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = nagoya_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -23.20713
target_lon = -45.86174

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in nagoya_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(nagoya_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in nagoya_points]
lons = [p[1] for p in nagoya_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(nagoya_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")